In [ ]:
!pip install timm coremltools yacs loguru smplx -q

In [ ]:
!git clone --recursive https://github.com/yohanshin/WHAM.git

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import glob
import os

IMG_DIR = '/kaggle/input/datasets/quocbao8925/coco-person-only/images/train/' 
BATCH_SIZE = 32
EPOCHS = 5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import sys
import os

for key in list(sys.modules.keys()):
    if key == 'configs' or key.startswith('configs.'):
        del sys.modules[key]

wham_path = '/kaggle/working/WHAM'
if wham_path in sys.path:
    sys.path.remove(wham_path)
sys.path.insert(0, wham_path)

if os.path.exists(wham_path):
    os.chdir(wham_path)
else:
    raise FileNotFoundError("WHAM folder not found. Check your clone path.")

import torch
from lib.models.preproc.backbone.hmr2 import hmr2

In [ ]:
!mkdir checkpoints
!gdown "https://drive.google.com/uc?id=1J6l8teyZrL0zFzHhzkC7efRhU0ZJ5G9Y&export=download&confirm=t" -O 'checkpoints/hmr2a.ckpt'

In [ ]:
!mkdir -p dataset/body_models
!gdown --id 1pbmzRbWGgae6noDIyQOnohzaVnX_csUZ -O dataset/body_models.tar.gz
!tar -xvf dataset/body_models.tar.gz -C dataset/

In [ ]:
print("Loading the real teacher model...")
teacher = hmr2('/kaggle/working/checkpoints/hmr2a.ckpt').to(DEVICE)
teacher.eval()
for param in teacher.parameters():
    param.requires_grad = False
print("Teacher loaded cleanly. Proceed with distillation.")

In [ ]:
import torch
import torch.nn as nn
import timm
from torch.utils.data import Dataset
import glob
import os
from PIL import Image

class FastViTStudent(nn.Module):
    def __init__(self, model_name='fastvit_sa24', target_dim=1024):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=True, num_classes=0)
        
        with torch.no_grad():
            dummy_input = torch.randn(1, 3, 224, 224)
            student_out_dim = self.backbone(dummy_input).shape[1]
        
        self.proj = nn.Linear(student_out_dim, target_dim)

    def forward(self, x):
        features = self.backbone(x)
        return self.proj(features)

print("Instantiating the student...")
student = FastViTStudent(target_dim=1024).to(DEVICE)
print("Student ready to learn.")

In [ ]:
transform = T.Compose([
    T.Resize((256, 256), antialias=True),
    T.Lambda(lambda x: x.float() / 255.0),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class FastDistillDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_paths = glob.glob(os.path.join(img_dir, '*.jpg'))
        self.transform = transform
        print(f"Đã nạp {len(self.img_paths)} ảnh.")

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        image = io.read_image(self.img_paths[idx], mode=io.ImageReadMode.RGB)
        if self.transform:
            image = self.transform(image)
        return image

In [ ]:
dataset = FastDistillDataset(img_dir=IMG_DIR, transform=transform)

train_loader = DataLoader(
    dataset, 
    batch_size=32, 
    shuffle=True, 
    num_workers=4,
    prefetch_factor=2,
    pin_memory=True
)

print(f"Train loader đã sẵn sàng với {len(train_loader)} batches.")

In [ ]:
from tqdm import tqdm
import torch
import torchvision.io as io

student = FastViTStudent().to(DEVICE)
optimizer = torch.optim.AdamW(student.parameters(), lr=1e-4)
criterion = torch.nn.MSELoss()

print("Starting the distillation. Do not interrupt this unless things are actually on fire.")
for epoch in range(EPOCHS):
    student.train()
    running_loss = 0.0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for images in pbar:
        images = images.to(DEVICE)
        
        with torch.no_grad():
            t_feat = teacher(images, encode=True) 
            
        s_feat = student(images)
        
        loss = criterion(s_feat, t_feat)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
        
    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} finished. Avg Loss: {avg_loss:.4f}")

torch.save(student.state_dict(), '/kaggle/working/fastvit_student_1024.pth')
print("Saved. Now export THIS to CoreML and your iPhone pipeline will stop crashing.")